In [ ]:

import scanpy as sc
import torch
import os
import pandas as pd
os.chdir("/public/home/off_liukunpeng/project/11_cluster_problem/AgaeSMO")


In [ ]:
import sys
sys.path.append("/public/home/off_liukunpeng/project/11_cluster_problem/AgaeSMO")

In [ ]:

import community as louvain
import AgaeSMO as AgaeSMO_v1
random_seed = 2022
AgaeSMO_v1.fix_seed(random_seed)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


In [ ]:
os.environ['R_HOME'] = '/public/home/off_liukunpeng/software/anaconda3/envs/pyg1/lib/R' 


In [ ]:

adata_omics1 = sc.read_h5ad("data/Human_Lymph_Node_A1/adata_RNA.h5ad")
adata_omics2 = sc.read_h5ad("data/Human_Lymph_Node_A1/adata_ADT.h5ad")

adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()

In [ ]:
sc.pp.filter_genes(adata_omics1, min_cells=10)
sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=3000)
sc.pp.normalize_total(adata_omics1, target_sum=1e4)
sc.pp.log1p(adata_omics1)
sc.pp.scale(adata_omics1)

In [ ]:
adata_omics1_high =  adata_omics1[:, adata_omics1.var['highly_variable']]
adata_omics1.obsm['feat'] = AgaeSMO_v1.pca(adata_omics1_high, n_comps=adata_omics2.n_vars-1)
adata_omics1.obsm['tensor']=adata_omics1[:, adata_omics1.var['highly_variable']].X
# adata_omics1.obsm['tensor']=AgaeSMO_v1.pca(adata_omics1_high, n_comps=100)

adata_omics2 = AgaeSMO_v1.clr_normalize_each_cell(adata_omics2)
sc.pp.scale(adata_omics2)
adata_omics2.obsm['feat'] = AgaeSMO_v1.pca(adata_omics2, n_comps=adata_omics2.n_vars-1)
# adata_omics2.obsm['tensor']=AgaeSMO_v1.pca(adata_omics2, n_comps=adata_omics2.n_vars-1)
adata_omics2.obsm['tensor']=adata_omics2.X


In [ ]:
data = AgaeSMO_v1.construct_neighbor_graph(adata_omics1, adata_omics2,6,6,k=6)


In [ ]:
model = AgaeSMO_v1.Train_AgaeSMO(data, device=device,learning_rate=0.001,epochs=600,loss_weight=[1,1,1,1])
output = model.train()

In [ ]:
adata = adata_omics1.copy()
adata.obsm['emb_latent_omics1'] = output['emb_latent_omics1'].copy()
adata.obsm['emb_latent_omics2'] = output['emb_latent_omics2'].copy()
adata.obsm['AgaeSMO'] = output['AgaeSMO'].copy()
adata.obsm['alpha'] = output['alpha']

In [ ]:

n_cluster=6
tool = 'mclust' # mclust, leiden, and louvain  
AgaeSMO_v1.clustering(adata,refine_=False, key='AgaeSMO', add_key='AgaeSMO', n_clusters=n_cluster, method=tool, use_pca=True)# visualization


annotation=pd.read_csv('data/Human_Lymph_Node_A1/annotation.csv')

adata.obs['Ground Truth']=annotation.loc[:,'manual-anno'].to_list()
indexs=AgaeSMO_v1.supervise_index(adata,"AgaeSMO",'Ground Truth')
print(indexs)

ARI=indexs["ARI"]

import matplotlib.pyplot as plt

fig, ax_list = plt.subplots(1, 2, figsize=(12, 4))
sc.pp.neighbors(adata, use_rep='AgaeSMO', n_neighbors=10)
sc.tl.umap(adata)

sc.pl.umap(adata, color='AgaeSMO', ax=ax_list[0], title='AgaeSMO', s=30, show=False)
sc.pl.embedding(adata, basis='spatial', color='AgaeSMO', ax=ax_list[1], title=f'AgaeSMO:{ARI}', s=50, show=False)

plt.tight_layout(w_pad=0.3)
# plt.show()


